# I-JEPA Region Embedding Extraction

This notebook submits SLURM jobs to extract region embeddings using ViT patch tokens pooled per SAM mask (REN-style).

## Supported Backbones:
- **I-JEPA ViT-H/14** (1280-dim) - Main experiment
- **DINOv2 ViT-L/14** (1024-dim) - Comparison
- **DINO ViT-B/8** (768-dim) - Ablation

## Configuration

Set your backbone and paths below:

In [ ]:
import os
import subprocess
from pathlib import Path

# ──────────── Configuration ────────────
BACKBONE = "ijepa_vit_h14"  # Options: ijepa_vit_h14 | dinov2_vitl14 | dino_vitb8

# Paths
REPO_ROOT = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen"
SUBMISSION_SCRIPT = f"{REPO_ROOT}/scripts/submit_region_extraction.sh"

# Output directories (based on backbone)
OUTPUT_DIRS = {
    "ijepa_vit_h14": f"{REPO_ROOT}/sam_cache_ijepa_h100",
    "dinov2_vitl14": f"{REPO_ROOT}/sam_cache_dinov2",
    "dino_vitb8": f"{REPO_ROOT}/sam_cache_dino",
}

OUTPUT_DIR = OUTPUT_DIRS[BACKBONE]

print(f"✓ Backbone: {BACKBONE}")
print(f"✓ Output: {OUTPUT_DIR}")
print(f"✓ Submission script: {SUBMISSION_SCRIPT}")

## Submit Extraction Job

Run this cell to submit the SLURM job:

In [ ]:
# Submit the SLURM job with the selected backbone
env = os.environ.copy()
env["BACKBONE"] = BACKBONE

result = subprocess.run(
    ["sbatch", SUBMISSION_SCRIPT],
    env=env,
    capture_output=True,
    text=True,
    cwd=REPO_ROOT
)

print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode == 0:
    # Extract job ID from "Submitted batch job XXXXXX"
    if "Submitted batch job" in result.stdout:
        job_id = result.stdout.split()[-1]
        print(f"\n✅ Job submitted successfully!")
        print(f"Job ID: {job_id}")
        print(f"\nMonitor with: squeue -j {job_id}")
        print(f"Cancel with: scancel {job_id}")
    else:
        print("\n✅ Command executed successfully")
else:
    print(f"\n❌ Error: Return code {result.returncode}")

## Monitor Job Status

Check the status of your running/pending jobs:

In [ ]:
!squeue -u $USER | grep region_emb

## View Log Files

Check the output and error logs:

In [ ]:
# Show last 50 lines of output log
LOG_DIR = f"{REPO_ROOT}/scripts/SEG-RDM/rdm/SLRUM_OUTPUT_FILES"
print("=" * 80)
print("OUTPUT LOG (last 50 lines):")
print("=" * 80)
!tail -n 50 {LOG_DIR}/region_emb_extract-h100.out

In [ ]:
# Show last 50 lines of error log
print("=" * 80)
print("ERROR LOG (last 50 lines):")
print("=" * 80)
!tail -n 50 {LOG_DIR}/region_emb_extract-h100.err

## Check Extraction Progress

Monitor how many embeddings have been extracted:

In [ ]:
import glob
from collections import defaultdict

# Count extracted .npz files by class
class_counts = defaultdict(int)
total_files = 0

npz_pattern = f"{OUTPUT_DIR}/*/masks_npz/*.npz"
npz_files = glob.glob(npz_pattern)

for npz_file in npz_files:
    class_id = Path(npz_file).parent.parent.name
    class_counts[class_id] += 1
    total_files += 1

print(f"Total extracted: {total_files} files")
print(f"Classes with data: {len(class_counts)}")

if class_counts:
    sorted_classes = sorted(class_counts.items(), key=lambda x: int(x[0]) if x[0].isdigit() else x[0])
    print(f"\nFirst 10 classes:")
    for class_id, count in sorted_classes[:10]:
        print(f"  Class {class_id}: {count} files")
    
    avg_per_class = total_files / len(class_counts)
    print(f"\nAverage per class: {avg_per_class:.1f} files")
else:
    print("\nNo extracted files found yet. Check if job is running.")

## Embedding Quality Analysis

Once extraction completes, analyze cosine similarity to verify diversity:

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def analyze_embedding_quality(output_dir, num_classes=5):
    """Analyze cosine similarity of extracted embeddings."""
    
    for class_id in range(num_classes):
        print(f"\n{'='*60}")
        print(f"CLASS {class_id}")
        print(f"{'='*60}")
        
        similarities = []
        total_files = 0
        
        pattern = f'{output_dir}/{class_id}/masks_npz/*.npz'
        npz_files = glob.glob(pattern)
        
        if len(npz_files) == 0:
            print(f"No files found for class {class_id}")
            continue
        
        for npz_file in npz_files[:50]:  # Sample first 50 files per class
            total_files += 1
            data = np.load(npz_file)
            
            if 'emb' not in data:
                continue
                
            emb = data['emb']
            
            if emb.shape[0] > 1:
                # Cosine similarity between all pairs of regions
                sim_matrix = cosine_similarity(emb)
                triu_indices = np.triu_indices_from(sim_matrix, k=1)
                similarities.append(sim_matrix[triu_indices].mean())
        
        if len(similarities) == 0:
            print(f"No valid embeddings found")
            continue
        
        sims = np.array(similarities)
        print(f"Files analyzed: {total_files}")
        print(f"\nAverage cosine similarity: {np.mean(sims):.3f}")
        print(f"Std dev: {np.std(sims):.3f}")
        print(f"Min: {np.min(sims):.3f}, Max: {np.max(sims):.3f}")
        
        print(f"\nSimilarity distribution:")
        print(f"  < 0.4 (good diversity): {100*np.sum(sims < 0.4)/len(sims):.1f}%")
        print(f"  0.4-0.6 (moderate):     {100*np.sum((sims >= 0.4) & (sims < 0.6))/len(sims):.1f}%")
        print(f"  0.6-0.8 (high sim):     {100*np.sum((sims >= 0.6) & (sims < 0.8))/len(sims):.1f}%")
        print(f"  > 0.8 (collapsed):      {100*np.sum(sims >= 0.8)/len(sims):.1f}%")

# Run analysis (uncomment when extraction is complete)
# analyze_embedding_quality(OUTPUT_DIR, num_classes=5)

## Compare Backbones

Extract embeddings with different backbones and compare their diversity:

In [ ]:
# Submit jobs for all three backbones
backbones = ["ijepa_vit_h14", "dinov2_vitl14", "dino_vitb8"]

for backbone in backbones:
    env = os.environ.copy()
    env["BACKBONE"] = backbone
    
    result = subprocess.run(
        ["sbatch", SUBMISSION_SCRIPT],
        env=env,
        capture_output=True,
        text=True,
        cwd=REPO_ROOT
    )
    
    if "Submitted batch job" in result.stdout:
        job_id = result.stdout.split()[-1]
        print(f"✓ {backbone}: Job {job_id}")
    else:
        print(f"✗ {backbone}: {result.stdout}")

## Cancel Job

If you need to cancel the running job:

In [ ]:
# Replace XXXXX with your job ID
JOB_ID = "XXXXX"  

# Uncomment to cancel:
# !scancel {JOB_ID}
# print(f"Cancelled job {JOB_ID}")